# Ch.02-02 데이터 전처리

**노드:** N10 train_test_split · N11 스케일링 · N12 (결측/인코딩은 다음)

1. **train / test 분리** — 모의고사
2. **스케일링** — feature 단위가 다를 때 거리 왜곡 (Ch.1 k-NN)
3. **올바른 순서** — split → **train으로만** fit → train·test transform


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(42)


## 1) 샘플 데이터 — 생선 (길이 cm · 무게 g)

In [ ]:
X = np.array([
    [25, 150], [26, 145], [24, 155], [27, 148],   # 빙어
    [28, 200], [30, 180], [29, 210], [31, 195],   # 도미
    [25, 152], [26, 140], [28, 205], [30, 190],
])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1])  # 0=빙어, 1=도미
feature_names = ["길이 (cm)", "무게 (g)"]
print("X.shape:", X.shape)


## 2) train_test_split — **연습 vs 모의고사**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("train:", X_train.shape[0], "마리 · test:", X_test.shape[0], "마리")
print("y_train:", y_train, "\ny_test:", y_test)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=["#5B9BD5" if v == 0 else "#ED7D31" for v in y_train],
           s=120, marker="o", edgecolors="white", linewidths=2, label="train (학습)")
ax.scatter(X_test[:, 0], X_test[:, 1], c=["#5B9BD5" if v == 0 else "#ED7D31" for v in y_test],
           s=180, marker="s", edgecolors="black", linewidths=2, label="test (모의고사)")
ax.set_xlabel(feature_names[0])
ax.set_ylabel(feature_names[1])
ax.set_title("train_test_split — 동그라미=train, 네모=test")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3) 스케일 문제 — **무게 차이가 거리를 지배**

길이 3cm 차이 vs 무게 50g 차이 — 숫자 크기가 다르면 k-NN은 **무게만** 본다.

In [ ]:
a = np.array([25, 150])
b = np.array([28, 150])  # 길이만 3cm 다름
c = np.array([25, 200])  # 무게만 50g 다름

def dist(p, q):
    return np.sqrt(((p - q) ** 2).sum())

print(f"길이 3cm 차이 거리: {dist(a, b):.1f}")
print(f"무게 50g 차이 거리: {dist(a, c):.1f}")
print("→ 스케일 안 맞추면 '50'이 '3'보다 훨씬 멀리 느껴짐")


## 4) StandardScaler — **train 평균·표준편차로 맞춤**

$$z = \frac{x - \mu_{train}}{\sigma_{train}}$$

**규칙:** `scaler.fit(X_train)` 만 · test 정보는 **절대** fit에 넣지 않음 (모의고사 답 leak)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # train에서 구한 μ, σ만 사용

print("train 원본 mean:", X_train.mean(axis=0))
print("train 원본 std :", X_train.std(axis=0, ddof=0))
print("scaled train mean (~0):", X_train_scaled.mean(axis=0).round(3))
print("scaled train std  (~1):", X_train_scaled.std(axis=0, ddof=0).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, data, title in [
    (axes[0], X_train, "전처리 전 (원본 단위)"),
    (axes[1], X_train_scaled, "StandardScaler 후 (z-score)"),
]:
    ax.scatter(data[:, 0], data[:, 1], c=["#5B9BD5" if v == 0 else "#ED7D31" for v in y_train], s=100, edgecolors="white")
    ax.set_xlabel(feature_names[0] if "전" in title else "길이 (scaled)")
    ax.set_ylabel(feature_names[1] if "전" in title else "무게 (scaled)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal", adjustable="box")

plt.suptitle("스케일링 후 두 축이 비슷한 범위 → 거리가 공정해짐", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## 5) k-NN — 스케일링 전 vs 후 **test 정확도**

In [ ]:
def acc(y_true, y_pred):
    return (y_true == y_pred).mean()

knn_raw = KNeighborsClassifier(n_neighbors=3)
knn_raw.fit(X_train, y_train)
pred_raw = knn_raw.predict(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=3)
knn_scaled.fit(X_train_scaled, y_train)
pred_scaled = knn_scaled.predict(X_test_scaled)

print("test accuracy (스케일링 X):", acc(y_test, pred_raw))
print("test accuracy (StandardScaler):", acc(y_test, pred_scaled))


## 6) 전처리 파이프라인 (기억할 순서)

```text
1. train_test_split(X, y)
2. scaler.fit(X_train)          ← train만
3. X_train_s = transform(train)
4. X_test_s  = transform(test)  ← 같은 scaler
5. model.fit(X_train_s, y_train)
6. model.predict(X_test_s)
```

**다음 (02-3):** 결측치 · 원-핫 인코딩 · `Pipeline`